In [2]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid
from datetime import date

task_id = 50
parent_run_id = ""
task_executions_id = ""
lineage_id = str(uuid.uuid4())
source_connection_settings = "{}"
source_settings = "{}"
target_settings = "{}"
limit_rows_for_debugging = False

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, 4, Finished, Available, Finished, False)

In [3]:
%run nb_transactional_shared_functions

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, 9, Finished, Available, Finished, True)

In [4]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, 10, Finished, Available, Finished, False)

In [5]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_schema = source_settings.get("schema_name", "silver_transactional")
source_table = source_settings.get("table_name", "silver_transactional_chrono_metrics")

# Target configuration
target_schema = target_settings.get("schema_name", "gold_transactional")
target_table = target_settings.get("table_name", "gold_transactional_chrono_metrics")

# Use framework lineage when present; keep the old fallback for manual runs.
effective_lineage_id = lineage_id or "44444444-4444-4444-aaaa-aaaaaaaaaaaa"

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, 11, Finished, Available, Finished, False)

In [6]:
# SQL Logic
df = spark.sql(f"""
SELECT DISTINCT
    CAST(-1 AS BIGINT) AS chronometrics_key,
    CAST(timecard_id AS STRING) AS timecard_id,
    CAST(client_code AS STRING) AS client_number,
    CAST(client_code AS STRING) AS client_key,
    CAST(matter_code AS STRING) AS matter_number,
    CAST(matter_region_src AS STRING) AS member_firm_code,
    CAST(matter_code AS STRING) AS matter_key,
    CAST(group_matter_code AS STRING) AS group_matter_number,
    CAST(group_matter_region_src AS STRING) AS group_matter_member_firm_code,
    CAST(group_matter_code AS STRING) AS group_matter_key,
    CAST(matter_company_code AS STRING) AS matter_company_key,
    CAST(matter_billable_type_code AS STRING) AS matter_billable_type_code,
    CAST(matter_currency_code AS STRING) AS matter_currency_code,
    CAST(timekeeper_id AS STRING) AS timekeeper_id,
    CAST(fe_region_src_tran AS STRING) AS timekeeper_member_firm_code,
    CAST(timekeeper_id AS STRING) AS fe_person_key,
    CAST(fe_company_code AS STRING) AS fe_company_key,
    COALESCE(NULLIF(TRIM(CAST(fe_office_code AS STRING)), ''), 'Unknown') AS fe_office_key,
    COALESCE(NULLIF(TRIM(CAST(fe_practice_group_code AS STRING)), ''), 'Unknown') AS fe_practice_group_key,
    COALESCE(NULLIF(TRIM(CAST(fe_team_code AS STRING)), ''), 'Unknown') AS fe_team_key,
    COALESCE(NULLIF(TRIM(CAST(fe_company_code_tran AS STRING)), ''), 'Unknown') AS fe_company_tran_key,
    COALESCE(NULLIF(TRIM(CAST(fe_office_code_tran AS STRING)), ''), 'Unknown') AS fe_office_tran_key,
    COALESCE(NULLIF(TRIM(CAST(fe_practice_group_code_tran AS STRING)), ''), 'Unknown') AS fe_practice_group_tran_key,
    COALESCE(NULLIF(TRIM(CAST(fe_group_code AS STRING)), ''), 'Unknown') AS fe_team_tran_key,
    COALESCE(NULLIF(TRIM(CAST(fe_sub_group_code AS STRING)), ''), 'Unknown') AS fe_employee_sub_group_key,
    COALESCE(NULLIF(TRIM(CAST(fe_band_level_code AS STRING)), ''), 'Unknown') AS fe_band_level_key,
    CAST(timecard_worked_date AS STRING) AS timecard_worked_date,
    CAST(timecard_worked_cal_period AS STRING) AS timecard_worked_cal_period,
    CAST(timecard_post_date AS STRING) AS timecard_post_date,
    CAST(timecard_post_cal_period AS STRING) AS timecard_post_cal_period,
    CAST(confidential_flag AS STRING) AS confidential_flag,
    CAST(pricing_unit AS STRING) AS pricing_unit,
    CAST(hours_worked AS DOUBLE) AS hours_worked,
    COALESCE(NULLIF(TRIM(CAST(timecard_narrative AS STRING)), ''), 'Unknown') AS timecard_narrative,
    CAST(worked_standard_rate_dc AS DOUBLE) AS worked_standard_rate_dc,
    CAST(fees_worked_at_standard_dc AS DOUBLE) AS fees_worked_at_standard_dc,
    CAST(worked_agreed_rate_dc AS DOUBLE) AS worked_agreed_rate_dc,
    CAST(fees_worked_at_agreed_dc AS DOUBLE) AS fees_worked_at_agreed_dc,
    CAST(matter_discount_dc AS DOUBLE) AS matter_discount_dc,
    CAST(wip_hours_write_up_down_value_dc AS DOUBLE) AS wip_hours_write_up_down_value_dc,
    CAST(fees_worked_at_agreed_net_dc AS DOUBLE) AS fees_worked_at_agreed_net_dc,
    CAST(fees_billed_at_standard_dc AS DOUBLE) AS fees_billed_at_standard_dc,
    CAST(fees_billed_at_agreed_dc AS DOUBLE) AS fees_billed_at_agreed_dc,
    CAST(matter_discount_billed_dc AS DOUBLE) AS matter_discount_billed_dc,
    CAST(fees_billed_at_agreed_net_dc AS DOUBLE) AS fees_billed_at_agreed_net_dc,
    CAST(fees_write_up_down_dc AS DOUBLE) AS fees_write_up_down_dc,
    CAST(hours_write_up_down_value_dc AS DOUBLE) AS hours_write_up_down_value_dc,
    CAST(worked_fees_billed_net_dc AS DOUBLE) AS worked_fees_billed_net_dc,
    CAST(bill_num AS STRING) AS bill_num,
    COALESCE(NULLIF(TRIM(CAST(cancel_bill_num AS STRING)), ''), 'Unknown') AS cancel_bill_num,
    COALESCE(NULLIF(TRIM(CAST(bill_type AS STRING)), ''), 'Unknown') AS bill_type_key,
    CAST(bill_date AS STRING) AS bill_date,
    CAST(bill_cal_period AS STRING) AS bill_cal_period,
    COALESCE(NULLIF(TRIM(CAST(bill_currency_code AS STRING)), ''), 'Unknown') AS bill_currency_code,
    CAST(hours_billed AS DOUBLE) AS hours_billed,
    CAST(fees_billed_dc AS DOUBLE) AS fees_billed_dc,
    CAST(bad_debt_wo_dc AS DOUBLE) AS bad_debt_wo_dc,
    CAST(final_bill_num AS STRING) AS final_bill_num,
    CAST(final_bill_flag AS STRING) AS final_bill_flag,
    CAST(material_group AS STRING) AS entry_classification_key,
    CAST(collected_cal_period AS STRING) AS collected_cal_period,
    CAST(collected_date AS STRING) AS collected_date,
    CAST(hours_collected AS DOUBLE) AS hours_collected,
    CAST(fees_collected_dc AS DOUBLE) AS fees_collected_dc,
    COALESCE(NULLIF(TRIM(CAST(phase_set_code AS STRING)), ''), 'Unknown') AS phase_set_code,
    COALESCE(NULLIF(TRIM(CAST(phase_set_desc AS STRING)), ''), 'Unknown') AS phase_set_description,
    COALESCE(NULLIF(TRIM(CAST(phase_code AS STRING)), ''), 'Unknown') AS phase_code,
    COALESCE(NULLIF(TRIM(CAST(phase_desc AS STRING)), ''), 'Unknown') AS phase_description,
    COALESCE(NULLIF(TRIM(CAST(task_code AS STRING)), ''), 'Unknown') AS task_code,
    COALESCE(NULLIF(TRIM(CAST(task_desc AS STRING)), ''), 'Unknown') AS task_description,
    COALESCE(NULLIF(TRIM(CAST(activity_code AS STRING)), ''), 'Unknown') AS activity_code,
    COALESCE(NULLIF(TRIM(CAST(activity_desc AS STRING)), ''), 'Unknown') AS activity_description,
    COALESCE(NULLIF(TRIM(CAST(fixed_fee_activity_code AS STRING)), ''), 'Unknown') AS fixed_fee_activity_code,
    COALESCE(NULLIF(TRIM(CAST(fixed_fee_activity_desc AS STRING)), ''), 'Unknown') AS fixed_fee_activity_description,
    TO_TIMESTAMP(last_update_dt, 'dd/MM/yyyy') AS last_updated_date_utc_at_source,

--metadata
    CAST(_ingest_date AS DATE) AS _ingest_date,
    CAST(_source_file AS STRING) AS _source_file,
    CAST(_source_system AS STRING) AS _source_system,
    '{target_table}' AS _table,
    CAST('{effective_lineage_id}' AS STRING) AS _lineage_id

--hashkey
    md5(concat_ws('|',
        coalesce(cast(timecard_id as string), ''),
        coalesce(cast(client_code as string), ''),
        coalesce(cast(matter_code as string), ''),
        coalesce(cast(matter_region_src as string), ''),
        coalesce(cast(group_matter_code as string), ''),
        coalesce(cast(group_matter_region_src as string), ''),
        coalesce(cast(timekeeper_id as string), ''),
        coalesce(cast(timecard_worked_date as string), ''),
        coalesce(cast(timecard_post_date as string), ''),
        coalesce(cast(bill_num as string), ''),
        coalesce(cast(final_bill_num as string), ''),
        coalesce(cast(hours_worked as string), ''),
        coalesce(cast(collected_date as string), ''),
        coalesce(cast(phase_code as string), ''),
        coalesce(cast(phase_desc as string), ''),
        coalesce(cast(task_code as string), ''),
        coalesce(cast(task_desc as string), '')
    )) AS _hashed_pk
FROM {source_schema}.{source_table}
""")

display(df)

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, 12, Finished, Available, Finished, False)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'md5'.(line 87, pos 4)

== SQL ==

SELECT DISTINCT
    CAST(-1 AS BIGINT) AS chronometrics_key,
    CAST(timecard_id AS STRING) AS timecard_id,
    CAST(client_code AS STRING) AS client_number,
    CAST(client_code AS STRING) AS client_key,
    CAST(matter_code AS STRING) AS matter_number,
    CAST(matter_region_src AS STRING) AS member_firm_code,
    CAST(matter_code AS STRING) AS matter_key,
    CAST(group_matter_code AS STRING) AS group_matter_number,
    CAST(group_matter_region_src AS STRING) AS group_matter_member_firm_code,
    CAST(group_matter_code AS STRING) AS group_matter_key,
    CAST(matter_company_code AS STRING) AS matter_company_key,
    CAST(matter_billable_type_code AS STRING) AS matter_billable_type_code,
    CAST(matter_currency_code AS STRING) AS matter_currency_code,
    CAST(timekeeper_id AS STRING) AS timekeeper_id,
    CAST(fe_region_src_tran AS STRING) AS timekeeper_member_firm_code,
    CAST(timek

In [ ]:
# Validate duplicates from shared_functions
validate_duplicates(df, "_hashed_pk", 10)

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, -1, Cancelled, , Cancelled, True)

In [ ]:
# Load to target from shared_functions
load_to_target(df, location_path, "merge")

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, -1, Cancelled, , Cancelled, True)

In [ ]:
# Validation
df = spark.sql(f"""
SELECT COUNT(*) FROM {target_schema}.{target_table}
""")
display(df)

StatementMeta(, 139a47bf-d285-4dd3-85be-89dd51e48aaa, -1, Cancelled, , Cancelled, True)